# P5a · Socioeconomic Characterization of Rural Demographic Groups
## *RURIMESCAPE — Paper 1, Step 5a*

---

**Paper:** Rural Migration and Land Use in Spain — Paper 1  
**Step:** 5a — Socioeconomic characterization and descriptive comparative analysis  
**Author:** Juan Zotes  
**Last updated:** 2026-04

---

### Context and purpose

Building on the behavioural classification and spatial analysis from Steps 3 and 4,
this notebook characterises the **socioeconomic conditions** associated with the four
demographic behavioural groups identified for Spanish rural municipalities:

| Group | Definition |
|-------|------------|
| **Grows in both** | Sustained dynamisers — positive growth in Period A (2010-2017) and Period B (2018-2025) |
| **Reverses in B** | Reverters — declined in A, recovered in B (primary neo-rural signal) |
| **Loses in B** | Gained in A, declined in B |
| **Structural depopulation** | Declined in both periods — reference group for models in p5b |

The analysis covers **all six Goerlich typologies**. Paper 1 analysis focuses on
Rural-Remote and Rural-Accessible, filtered via DuckDB SQL at the point of use.
The full 8,132-municipality master dataset is preserved for Paper 2/3 reuse.

---

### ⚠️ Important note on variable temporality

SIDAMUN variables are from the **2023 reference snapshot** (downloaded February 2026).
They are **contemporaneous with or posterior to Period B (2018–2025)** and should be
interpreted as characterising the **current state** of each group, not as pre-turning-point
baseline conditions. This limitation is explicitly stated in the paper methodology section.

---

### Analytical structure

| Section | Content |
|---------|--------|
| 0 | Environment, paths, constants |
| 1 | **DuckDB setup** — first use of SQL in this project |
| 2 | Data loading and master dataset construction via DuckDB |
| 3 | Variable exploration: missings, candidate selection, Spearman correlation |
| 4 | Descriptive statistics — medians by group and typology |
| 5 | Kruskal-Wallis tests + Dunn post-hoc (Bonferroni correction) |
| 6 | Figures — boxplots by group and variable |
| 7 | Export outputs |
| 8 | Interpretation notes |

---

### Inputs

| File | Location | Description |
|------|----------|-------------|
| `p4_spatial_hotspots.gpkg` — layer `lisa_results_all` | `data/spatial/processed/` | **All 8,132 Spanish municipalities** with behavioural groups and LISA results (all 6 Goerlich typologies) |
| `SIDAMUN_MITERD.xlsx` | `data/services/raw/` | Socioeconomic variables by municipality (MITERD, 2023) |

### Join strategy

```
lisa_results_all  (8,132 rows, single load)
  LEFT JOIN SIDAMUN  →  master dataset (8,132 rows)

All typology filtering done via DuckDB SQL at point of use:
  WHERE tipo_goerlich = 'Rural - Remoto'          → Rural-Remote analysis
  WHERE tipo_goerlich = 'Rural - Accesible'        → Rural-Accessible analysis
  WHERE tipo_goerlich IN ('Rural - Remoto', ...)   → combined rural
  No filter                                        → all typologies (Paper 2/3)
```

### Outputs

| File | Location | Description |
|------|----------|-------------|
| `p5a_master_dataset.csv` | `data/demography/derived/paper1/` | **All 8,132 municipalities**, all typologies. Reusable for Paper 2/3. |
| `p5a_rural_analysis_dataset.csv` | `data/demography/derived/paper1/` | Rural subset (6,723 municipalities) filtered from master, selected variables. Input for p5b. |
| `p5a_selected_variables.csv` | `data/demography/derived/paper1/` | Variable selection log: missings, correlation with response, keep/drop decision |
| `p5a_descriptive_stats.csv` | `data/demography/derived/paper1/` | Medians by group × typology for all selected variables |
| `p5a_kruskal_wallis_results.csv` | `data/demography/derived/paper1/` | KW test results per variable and typology |
| `p5a_dunn_results.csv` | `data/demography/derived/paper1/` | Dunn post-hoc pairwise results (Bonferroni-corrected) |
| `figures/p5a/` | `figures/` | Boxplot figures per variable block |


---
## 0 · Environment, paths, constants

In [ ]:
"""
Notebook  : p5a_socioeconomic_characterization.ipynb
Author    : Juan Zotes
Created   : 2026-04

Purpose:
    Socioeconomic characterisation of the four rural demographic behavioural
    groups identified in p3/p4, using SIDAMUN variables (MITERD, 2023).

    Base dataset: lisa_results_all (8,132 municipalities, all Goerlich typologies).
    Typology filtering via DuckDB SQL — no separate layer loads needed.
    Paper 1 analysis focuses on Rural-Remote and Rural-Accessible.
    Master dataset (8,132 rows) reusable for Paper 2/3 without rebuilding.

    Methods:
        - Variable selection based on missings audit + Spearman correlation
        - Kruskal-Wallis test (non-parametric, no normality assumption)
        - Dunn post-hoc with Bonferroni correction for pairwise comparisons
        - Boxplot figures per thematic block

    NOTE: This notebook introduces DuckDB for the first time in this project.
    DuckDB is used for the multi-source join (GeoPackage + XLSX) and for
    all subsequent filtering and aggregation queries.

Inputs:
    - p4_spatial_hotspots.gpkg   layer: lisa_results_all   (spatial/processed)
    - SIDAMUN_MITERD.xlsx                                   (services/raw)

Outputs:
    - p5a_master_dataset.csv                 (demography/derived/paper1)
    - p5a_rural_analysis_dataset.csv         (demography/derived/paper1)
    - p5a_selected_variables.csv             (demography/derived/paper1)
    - p5a_descriptive_stats.csv              (demography/derived/paper1)
    - p5a_kruskal_wallis_results.csv         (demography/derived/paper1)
    - p5a_dunn_results.csv                   (demography/derived/paper1)
    - figures/p5a/                           (figures/)
"""

from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from scipy import stats
from scipy.stats import spearmanr
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')


In [ ]:
# ── Project root — adjust if project moves ────────────────────────────────────
ROOT = Path(r'C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain')

# ── Input paths ───────────────────────────────────────────────────────────────
GPKG_P4        = ROOT / 'data/spatial/processed/p4_spatial_hotspots.gpkg'
SIDAMUN_XLSX   = ROOT / 'data/services/raw/SIDAMUN_MITERD.xlsx'

# ── Output paths ──────────────────────────────────────────────────────────────
DEMO_DERIV     = ROOT / 'data/demography/derived/paper1'
FIG_DIR        = ROOT / 'figures/p5a_socioeconomic_characterization'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DEMO_DERIV.mkdir(parents=True, exist_ok=True)

# ── Typology constants ─────────────────────────────────────────────────────────
RURAL_REMOTE     = 'Rural - Remoto'
RURAL_ACCESSIBLE = 'Rural - Accesible'
RURAL_TYPES      = [RURAL_REMOTE, RURAL_ACCESSIBLE]

# ── Behavioural group constants and display order ─────────────────────────────
GROUP_ORDER = [
    'Grows in both',
    'Reverses in B',
    'Loses in B',
    'Structural depopulation'
]

GROUP_COLORS = {
    'Grows in both'          : '#d7191c',   # dark red
    'Reverses in B'          : '#fdae61',   # light orange
    'Loses in B'             : '#abd9e9',   # light blue
    'Structural depopulation': '#2c7bb6',   # dark blue
}

# ── SIDAMUN thematic blocks to load (excludes Demografía — already in p4) ────
# Block names correspond to the SIDAMUN header structure
SIDAMUN_BLOCKS = ['ECONOMIA', 'SERVICIOS', 'VIVIENDA', 'MEDIO FÍSICO', 'MEDIOAMBIENTE']

# Verify inputs exist
for p in [GPKG_P4, SIDAMUN_XLSX]:
    status = '✓' if p.exists() else '✗  NOT FOUND'
    print(f'  {status}  {p.name}')

---
## 1 · DuckDB setup

> **First use of DuckDB in this project.**
> 
> From this step onwards, multi-source data joins and filtering queries are handled
> via DuckDB — an in-process analytical SQL engine optimised for columnar operations.
> DuckDB replaces chained pandas merges with explicit, readable SQL that documents
> every join condition and column selection decision directly in the code.
> 
> **Why DuckDB here?**  
> This step joins three sources (GeoPackage layer + SIDAMUN XLSX + derived CSVs)
> with different schemas and potential column name collisions. SQL makes the join
> logic explicit and avoids accidental column duplication (`SELECT *` is never used
> in the final merge — every column is named).
>
> DuckDB operates in memory (no server needed) and integrates natively with pandas
> DataFrames. We use a persistent in-memory connection throughout this notebook.

In [ ]:
# ── Initialise DuckDB in-memory connection ────────────────────────────────────
# A single connection is reused throughout the notebook.
# Tables are registered as views from pandas DataFrames — no data is written to disk.

con = duckdb.connect()  # in-memory, no persistent file
print(f'DuckDB version: {duckdb.__version__}')
print('Connection established — in-memory mode.')

---
## 2 · Data loading and master dataset construction

In [ ]:
# ── 2.1 Load GeoPackage layer: lisa_results_all ──────────────────────────────
# Single layer load — all 8,132 Spanish municipalities, all 6 Goerlich typologies.
# Contains: Mun_Code, administrative hierarchy, tipo_goerlich, size_group,
#           behavioural_group, demographic period indicators, LISA results.
# Geometry column dropped — not needed for statistical analysis.
#
# All typology filtering (Rural-Remote, Rural-Accessible, etc.) is done
# downstream via DuckDB SQL — no separate layer loads needed.

print('Loading GeoPackage layer: lisa_results_all ...')
gdf_all = gpd.read_file(GPKG_P4, layer='lisa_results_all')
df_all = pd.DataFrame(gdf_all.drop(columns='geometry'))
df_all['Mun_Code'] = df_all['Mun_Code'].astype(str).str.zfill(5)

assert df_all['Mun_Code'].nunique() == len(df_all), 'Duplicate Mun_Code detected!'

print(f'Loaded: {len(df_all):,} municipalities')
print(f'Columns: {list(df_all.columns)}')
print()
print('Breakdown by typology and behavioural group:')
print(df_all.groupby(['tipo_goerlich', 'behavioural_group']).size().unstack(fill_value=0).to_string())


In [ ]:
# ── 2.2 Load SIDAMUN XLSX ──────────────────────────────────────────────────────
# SIDAMUN uses a 4-row multi-level header structure (block > subblock > variable > source/year).
# We load with header=None and manually flatten the column names to avoid ambiguity.
# Only non-demographic thematic blocks are retained:
#   ECONOMIA, SERVICIOS, VIVIENDA, MEDIO FÍSICO, MEDIOAMBIENTE
# The DEMOGRAFIA block is excluded — population data is already in the p4 layer
# with better temporal precision.

print('Loading SIDAMUN XLSX ...')
df_sidamun_raw = pd.read_excel(
    SIDAMUN_XLSX,
    header=[0, 1, 2, 3],   # 4-level multiindex header
    dtype=str              # Load everything as string first for inspection
)

print(f'SIDAMUN raw shape: {df_sidamun_raw.shape}')
print(f'Top-level blocks in header:')

# Extract top-level block names from the MultiIndex
top_blocks = df_sidamun_raw.columns.get_level_values(0).unique().tolist()
for b in top_blocks:
    n_cols = sum(1 for c in df_sidamun_raw.columns if c[0] == b)
    print(f'  {b}: {n_cols} columns')

In [ ]:
# ── 2.3 Flatten SIDAMUN column names and select ALL blocks (including DEMOGRAFIA) ──
# Flatten the 4-level MultiIndex into a single readable string per column.
# Format: BLOCK__Subblock__Variable (dropping 'Unnamed' levels).

def flatten_col(col_tuple):
    """Flatten a MultiIndex column tuple into a clean string key."""
    parts = [str(c).strip() for c in col_tuple if 'Unnamed' not in str(c) and str(c).strip()]
    return '__'.join(parts) if parts else 'unknown'

flat_cols = [flatten_col(c) for c in df_sidamun_raw.columns]
df_sidamun_raw.columns = flat_cols

# ── CRITICAL: use CMUNXL (5-digit province+municipality code) not CMUN (3-digit) ─
# CMUN  = 3 digits (municipal code only, e.g. '001') — WRONG for joining
# CMUNXL = 5 digits (province + municipal, e.g. '01001') — matches INE Mun_Code
mun_col = 'GENERAL__CMUNXL'
print(f'Mun_Code column: {mun_col}')

# ═══════════════════════════════════════════════════════════════════════════════
# UPDATED: Include DEMOGRAFIA block for sex, age groups, and aging indices
# ═══════════════════════════════════════════════════════════════════════════════
KEEP_BLOCKS = ['GENERAL', 'DEMOGRAFIA', 'ECONOMIA', 'SERVICIOS', 'VIVIENDA', 
               'MEDIO FÍSICO', 'MEDIOAMBIENTE']
cols_to_keep = [c for c in df_sidamun_raw.columns
                if any(c.startswith(b) for b in KEEP_BLOCKS)]
df_sidamun = df_sidamun_raw[cols_to_keep].copy()

# Skip metadata rows (INE source row + date row at top of SIDAMUN)
# First real data row: where CMUNXL contains a 5-digit numeric code
data_start = df_sidamun[mun_col].apply(
    lambda x: str(x).replace('.', '').isdigit() and len(str(x).replace('.', '').split('.')[0]) >= 4
).idxmax()
df_sidamun = df_sidamun.iloc[data_start:].reset_index(drop=True)

# Standardise Mun_Code: 5-digit zero-padded string
df_sidamun['Mun_Code'] = df_sidamun[mun_col].astype(str).str.split('.').str[0].str.zfill(5)

# Convert numeric columns from string to float
non_id_cols = [c for c in df_sidamun.columns if c != 'Mun_Code' and c != mun_col]
for col in non_id_cols:
    df_sidamun[col] = pd.to_numeric(df_sidamun[col], errors='coerce')

# Drop original CMUNXL column (replaced by standardised Mun_Code)
df_sidamun.drop(columns=[mun_col], inplace=True)

print(f'SIDAMUN cleaned: {df_sidamun.shape[0]:,} rows × {df_sidamun.shape[1]} cols')
print(f'Mun_Code sample: {df_sidamun["Mun_Code"].head(3).tolist()}')
print(f'Mun_Code unique: {df_sidamun["Mun_Code"].nunique():,} (expected 8,132)')
print(f'Format check (5 chars): {all(df_sidamun["Mun_Code"].str.len() == 5)}')


In [ ]:
# ── 2.3b Compute aggregated age groups from SIDAMUN quinquennial data ─────────
# SIDAMUN pyramid columns = % within each sex (not % of total population).
# e.g. Mujeres 15-19 = 8.5% means 8.5% of ALL women are aged 15-19.
# To convert to % of total population:
#   Pct_Mujeres_15_29 = Pirámide_Mujeres_15_29 (% s/ mujeres) × Pct_Mujeres_total / 100

import re

demo_cols = [c for c in df_sidamun.columns if c.startswith('DEMOGRAFIA')]

# ── Identify pyramid columns by sex ───────────────────────────────────────────
age_women_cols = [c for c in demo_cols
                  if 'PIRÁMIDE' in c and 'Mujeres' in c and '% hab. s/ total' in c]
age_men_cols   = [c for c in demo_cols
                  if 'PIRÁMIDE' in c and 'Hombres' in c and '% hab. s/ total' in c]

print(f'Pyramid columns: Mujeres={len(age_women_cols)}, Hombres={len(age_men_cols)}')

# ── Total sex weights (% of total population) ─────────────────────────────────
col_pct_mujeres = 'DEMOGRAFIA__POBLACIÓN POR SEXO__Mujeres \n(% hab. s/ total)'
col_pct_hombres = 'DEMOGRAFIA__POBLACIÓN POR SEXO__Hombres (% hab. s/ total)'

pct_mujeres = df_sidamun[col_pct_mujeres] / 100  # e.g. 0.4847
pct_hombres = df_sidamun[col_pct_hombres] / 100  # e.g. 0.5153

def extract_age_start(col_name):
    match = re.search(r'De (\d+) a', col_name)
    return int(match.group(1)) if match else None

def cols_for_range(col_list, start, end):
    return [c for c in col_list
            if extract_age_start(c) is not None
            and start <= extract_age_start(c) < end]

# ── Compute aggregated groups converted to % of total population ──────────────
AGE_RANGES = [('0_14', 0, 15), ('15_29', 15, 30), ('30_64', 30, 65), ('65_plus', 65, 999)]

for label, start, end in AGE_RANGES:
    w_cols = cols_for_range(age_women_cols, start, end)
    m_cols = cols_for_range(age_men_cols,   start, end)

    # % within sex × sex weight = % of total population
    pct_w = df_sidamun[w_cols].sum(axis=1) * pct_mujeres
    pct_m = df_sidamun[m_cols].sum(axis=1) * pct_hombres

    df_sidamun[f'DEMOGRAFIA__Pct_{label}']         = pct_w + pct_m
    df_sidamun[f'DEMOGRAFIA__Pct_Mujeres_{label}'] = pct_w
    df_sidamun[f'DEMOGRAFIA__Pct_Hombres_{label}'] = pct_m

# ── Quality control ───────────────────────────────────────────────────────────
df_sidamun['_age_total'] = sum(
    df_sidamun[f'DEMOGRAFIA__Pct_{label}'] for label, _, _ in AGE_RANGES
)

print(f'\nQuality control (should sum to ~100%):')
print(f'  Mean: {df_sidamun["_age_total"].mean():.2f}%')
print(f'  Min:  {df_sidamun["_age_total"].min():.2f}%')
print(f'  Max:  {df_sidamun["_age_total"].max():.2f}%')

outliers = df_sidamun[abs(df_sidamun['_age_total'] - 100) > 2]
if len(outliers) > 0:
    print(f'⚠ {len(outliers)} municipalities off by >2%')
else:
    print(f'✓ All municipalities pass QC')

df_sidamun.drop(columns=['_age_total'], inplace=True)

# ── Verification on first municipality ────────────────────────────────────────
mun = df_sidamun.iloc[0]
print(f'\nVerification mun[0]:')
print(f'  Pct_0_14:          {mun["DEMOGRAFIA__Pct_0_14"]:.2f}%')
print(f'  Pct_15_29:         {mun["DEMOGRAFIA__Pct_15_29"]:.2f}%')
print(f'  Pct_30_64:         {mun["DEMOGRAFIA__Pct_30_64"]:.2f}%')
print(f'  Pct_65_plus:       {mun["DEMOGRAFIA__Pct_65_plus"]:.2f}%')
total = (mun["DEMOGRAFIA__Pct_0_14"] + mun["DEMOGRAFIA__Pct_15_29"] +
         mun["DEMOGRAFIA__Pct_30_64"] + mun["DEMOGRAFIA__Pct_65_plus"])
print(f'  Suma:              {total:.2f}%  ← debe ser ~100%')
print(f'\n  Pct_Mujeres_15_29: {mun["DEMOGRAFIA__Pct_Mujeres_15_29"]:.2f}%')
print(f'  Pct_Hombres_15_29: {mun["DEMOGRAFIA__Pct_Hombres_15_29"]:.2f}%')
print(f'  Pct_Mujeres_65+:   {mun["DEMOGRAFIA__Pct_Mujeres_65_plus"]:.2f}%')
print(f'  Pct_Hombres_65+:   {mun["DEMOGRAFIA__Pct_Hombres_65_plus"]:.2f}%')

print(f'\nNew shape: {df_sidamun.shape}')

In [ ]:
# En una celda nueva temporal:
mun = df_sidamun.iloc[0]
print(f"Pct_0_14:    {mun['DEMOGRAFIA__Pct_0_14']:.2f}%")
print(f"Pct_15_29:   {mun['DEMOGRAFIA__Pct_15_29']:.2f}%")
print(f"Pct_30_64:   {mun['DEMOGRAFIA__Pct_30_64']:.2f}%")
print(f"Pct_65_plus: {mun['DEMOGRAFIA__Pct_65_plus']:.2f}%")
print(f"Suma:        {mun['DEMOGRAFIA__Pct_0_14'] + mun['DEMOGRAFIA__Pct_15_29'] + mun['DEMOGRAFIA__Pct_30_64'] + mun['DEMOGRAFIA__Pct_65_plus']:.2f}%")
# Si suma ~100% → los grupos están bien, el QC era incorrecto
# Si suma ~200% → hay que dividir entre 2

In [ ]:
# ── 2.4 Register DataFrames as DuckDB views ───────────────────────────────────
# Two views registered — one for each source.
# DuckDB queries these directly from memory; no disk writes.

con.register('all_mun', df_all)
con.register('sidamun', df_sidamun)

print('DuckDB views registered:')
print(f'  all_mun : {len(df_all):,} rows × {len(df_all.columns)} cols  (lisa_results_all)')
print(f'  sidamun : {len(df_sidamun):,} rows × {len(df_sidamun.columns)} cols  (SIDAMUN)')


In [ ]:
# ── 2.5 Build master dataset via DuckDB SQL ───────────────────────────────────
#
# Single LEFT JOIN: all_mun LEFT JOIN sidamun ON Mun_Code.
# Result: 8,132 rows — all municipalities, all typologies.
#
# Standard tabular key join — no spatial operations.
# NULL in SIDAMUN columns = statistical confidentiality suppression
# (common for smallest rural municipalities).
#
# No SELECT * — every column named explicitly to prevent duplication.
# SIDAMUN columns prefixed 'sid__' during join, cleaned afterwards.

sidamun_cols = [c for c in df_sidamun.columns if c != 'Mun_Code']
sidamun_select = ',\n    '.join([f's."{c}" AS "sid__{c}"' for c in sidamun_cols])

query = f"""
SELECT
    -- ── Identity and administrative hierarchy ────────────────────────────────
    a.Mun_Code,
    a.Mun_Name,
    a.Comarca_Code,
    a.Comarca_Name,
    a.Prov_Code,
    a.Prov_Name,
    a.CCAA_Code,
    a.CCAA_Name,

    -- ── Typology and size ────────────────────────────────────────────────────
    a.tipo_goerlich,
    a.size_group,
    a.Pop_ref,

    -- ── Demographic period indicators ───────────────────────────────────────
    a.pop_start_A,
    a.pop_end_A,
    a.var_acum_pct_A,
    a.var_anual_media_pct_A,
    a.pop_start_B,
    a.pop_end_B,
    a.var_acum_pct_B,
    a.var_anual_media_pct_B,

    -- ── Behavioural classification ───────────────────────────────────────────
    a.behavioural_group,

    -- ── LISA results ─────────────────────────────────────────────────────────
    a.lisa_I,
    a.lisa_p_sim,
    a.lisa_quadrant,
    a.lisa_quad_label,
    a.lisa_significant,
    a.lisa_sig_quad,

    -- ── SIDAMUN socioeconomic variables ──────────────────────────────────────
    {sidamun_select}

FROM all_mun a
LEFT JOIN sidamun s
    ON a.Mun_Code = s.Mun_Code
"""

df_master = con.execute(query).df()

print(f'Master dataset built: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns')
print(f'Row count check: {len(df_master):,} (expected 8,132) — {"✓" if len(df_master) == 8132 else "✗ MISMATCH"}')
print()
print('Breakdown by typology:')
print(df_master.groupby('tipo_goerlich').size().sort_values(ascending=False).to_string())


In [ ]:
# ── 2.6 Clean up column names: remove 'sid__' prefix ─────────────────────────
# The prefix was used to track SIDAMUN origin during the join.
# Now that the dataset is built and verified, rename to clean names.
# This MUST run before section 2.7 (imputation) and 3.1 (CANDIDATE_VARS).

rename_map = {c: c.replace('sid__', '') for c in df_master.columns if c.startswith('sid__')}
df_master.rename(columns=rename_map, inplace=True)

# Verify rename worked
remaining_sid = [c for c in df_master.columns if c.startswith('sid__')]
if remaining_sid:
    print(f'WARNING: {len(remaining_sid)} columns still have sid__ prefix')
else:
    print('Column rename complete — no sid__ prefixes remaining. ✓')

print(f'Master dataset: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns')
print(f'Sample columns: {[c for c in df_master.columns if "ECONOMIA" in c][:3]}')

In [ ]:
# ── 2.7 Recode and impute service variables ───────────────────────────────────
# All SI/NO columns in SIDAMUN 2023 are empty except three that store text 'SI'/'NO'.
# Rule:
#   - SI/NO with número equivalent → discard SI/NO, use número
#   - SI/NO without número equivalent → recode text/NULL to 1/0 and keep
#   - Count (número) columns → impute residual NULLs to 0
#
# Paper methodology note:
#   'Binary SI/NO columns with numeric equivalents were replaced by count variables.
#    SI/NO columns without numeric equivalents were recoded to binary 1/0.
#    Residual NULLs in count variables were imputed as 0 (absence of service).'

# ── Recode SI/NO-only variables to 1/0 ───────────────────────────────────────
SINO_RECODE = [
    'SERVICIOS__CULTURA__Disponibilidad de biblioteca  (SI/NO)',
    'SERVICIOS__CENTROS DE EDUCACIÓN__Municipios con escuela integrada en un centro rural agrupado  (SI/NO)',
    'SERVICIOS__CENTROS DE EDUCACIÓN__Centros de Enseñanzas de Rég. General No Universitarias (SI/NO)',
]

print('Recoding SI/NO-only variables to 1/0:')
for col in SINO_RECODE:
    if col in df_master.columns:
        df_master[col] = df_master[col].map(
            {'SI': 1, 'NO': 0, 'Sí': 1, 'No': 0, 1.0: 1, 0.0: 0}
        ).fillna(0).astype(int)
        n_ones = (df_master[col] == 1).sum()
        label = col.split('__')[-1]
        print(f'  ✓  {n_ones:,} with service (1)  |  {label}')
    else:
        print(f'  ✗  Not found: {col.split("__")[-1]}')

# ── Impute count variables: residual NULLs → 0 ───────────────────────────────
SERVICE_ZERO_IMPUTE = [
    'SERVICIOS__CENTROS SANITARIOS__Consultorio de atención primaria (número)',
    'SERVICIOS__ESTABLECIMIENTOS SANITARIOS__Oficina de farmacia (número)',
    'SERVICIOS__CENTROS DE EDUCACIÓN__Nº centros de Educación Infantil Segundo Ciclo',
    'SERVICIOS__CENTROS DE EDUCACIÓN__Nº centros de Educación Primaria',
    'SERVICIOS__OTROS SERVICIOS__SUCURSAL BANCARIA__Sucursal bancaria (número)',
]

print()
print('Imputing count variables (NULL → 0):')
for col in SERVICE_ZERO_IMPUTE:
    if col in df_master.columns:
        n_imputed = df_master[col].isna().sum()
        df_master[col] = df_master[col].fillna(0)
        label = col.split('__')[-1]
        print(f'  ✓  {n_imputed:,} NULLs → 0  |  {label}')
    else:
        print(f'  ✗  Not found: {col.split("__")[-1]}')

# ── Explicit exclusions ───────────────────────────────────────────────────────
EXPLICIT_EXCLUDE = [
    'MEDIOAMBIENTE__FACTORES DE VULNERABILIDAD Y RIESGOS NATURALES__ARIDEZ__Índice de aridez (clase)',
]
print()
print('Explicitly excluded (not imputable, not relevant):')
for col in EXPLICIT_EXCLUDE:
    print(f'  ✗  {col.split("__")[-1]}')

con.register('master', df_master)
out_master = DEMO_DERIV / 'p5a_master_dataset.csv'
df_master.to_csv(out_master, sep=';', encoding='utf-8-sig', index=False)
print()
print(f'Master dataset saved: {out_master.name}')
print(f'  {df_master.shape[0]:,} rows × {df_master.shape[1]} columns')

---
## 3 · Variable exploration and selection

Before running any statistical tests, we need empirical criteria for deciding
which SIDAMUN variables enter the analysis. Three filters are applied:

1. **Missing values audit** — variables with >40% missing in either typology are excluded.
   Small rural municipalities have many SIDAMUN values suppressed for statistical
   confidentiality reasons.

2. **Collinearity screening** — Spearman correlation matrix among candidates.
   Pairs with |ρ| > 0.85 are flagged; the less interpretable variable is dropped.

3. **Correlation with response** — Spearman correlation of each variable with
   the binary response variables (reverter and dynamiser vs. structural decline).
   This informs the variable selection for p5b but does not filter variables from
   the descriptive analysis (5a shows all retained variables regardless of signal strength).

All decisions are logged to `p5a_selected_variables.csv` for methodological transparency.

In [ ]:
demo_cols = [c for c in df_master.columns if 'DEMOGRAFIA' in c]
extran = [c for c in demo_cols if 'Extranjera' in c and '%' in c]
print(extran)


In [ ]:
# ── 3.1 Define candidate SIDAMUN variable blocks ─────────────────────────────
# SI/NO columns with número equivalent → replaced by número (count contains all info).
# SI/NO columns without número equivalent → recoded to 1/0 in section 2.7, kept here.
# Aridez excluded (genuine missing, not imputable).

# ═══════════════════════════════════════════════════════════════════════════════
# UPDATED: Added 'demographic' block with sex, age groups, and aging indices
# ═══════════════════════════════════════════════════════════════════════════════

CANDIDATE_VARS = {
    'demographic': [
        # ── Total sex ratio ───────────────────────────────────────────────────
        'DEMOGRAFIA__POBLACIÓN POR SEXO__Mujeres \n(% hab. s/ total)',
        'DEMOGRAFIA__Pct_0_14',
        'DEMOGRAFIA__Pct_15_29',
        'DEMOGRAFIA__Pct_30_64',
        'DEMOGRAFIA__Pct_65_plus',
        'DEMOGRAFIA__Pct_Mujeres_0_14',
        'DEMOGRAFIA__Pct_Hombres_0_14',
        'DEMOGRAFIA__Pct_Mujeres_15_29',
        'DEMOGRAFIA__Pct_Hombres_15_29',
        'DEMOGRAFIA__Pct_Mujeres_30_64',
        'DEMOGRAFIA__Pct_Hombres_30_64',
        'DEMOGRAFIA__Pct_Mujeres_65_plus',
        'DEMOGRAFIA__Pct_Hombres_65_plus',
        'DEMOGRAFIA__EDAD MEDIA__Edad media población',
        'DEMOGRAFIA__RATIOS__Índice envejecimiento\n(%)',
        'DEMOGRAFIA__RATIOS__Tasa dependencia \n(%)',
        'DEMOGRAFIA__NACIONALIDAD__Población Nacionalidad Extranjera \n(% hab. s/ total)',
    ],
    'economic': [
        'ECONOMIA__RENTAS__Renta neta media por persona',
        'ECONOMIA__RENTAS__Renta neta media por hogar',
        'ECONOMIA__RENTAS__DESIGUALDAD__Índice de Gini (%)',
        'ECONOMIA__RENTAS__DESIGUALDAD__Distribución de la renta P80/P20',
        'ECONOMIA__PARADOS__POR SEXO__Tasa de paro',
        'ECONOMIA__AFILIADOS__POR SECTOR__Afiliados Régimen Especial (R. E.) T. Autónomos \n(% s/ Total)',
        'ECONOMIA__CONTRATOS__POR DURACIÓN__Contratos indefinidos \n(% s/ Total)',
        'ECONOMIA__EMPRESAS__Total Empresas',
        'ECONOMIA__PENSIONES CONTRIBUTIVAS__Pensión Contributiva Media',
    ],
    # SI/NO replaced by número where available or deleted.
    # Biblioteca, CRA, Enseñanzas Rég. General kept as 1/0 (no número equivalent).
    'services': [
        'SERVICIOS__INTERNET__Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)',
        'SERVICIOS__CENTROS SANITARIOS__Consultorio de atención primaria (número)',
        'SERVICIOS__ESTABLECIMIENTOS SANITARIOS__Oficina de farmacia (número)',
        'SERVICIOS__CENTROS DE EDUCACIÓN__Nº centros de Educación Infantil Segundo Ciclo',
        'SERVICIOS__CENTROS DE EDUCACIÓN__Nº centros de Educación Primaria',
        'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 5.000 hab. o más, más cercano (minutos)',
        'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 20.000 hab. o más, más cercano (minutos)',
        'SERVICIOS__OTROS SERVICIOS__SUCURSAL BANCARIA__Sucursal bancaria (número)',
        'SERVICIOS__TRANSPORTE__Parque de vehículos x c/ 100 hab.',
    ],
    'housing': [
        'VIVIENDA__TIPOS DE VIVIENDAS (familiares)__Viviendas no principales (% s/ total)',
        'VIVIENDA__HOGAR__Tamaño medio del hogar',
        'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)',
        'VIVIENDA__VIVIENDAS TURÍSTICAS__Plazas turísticas x c/ 100 hab.',
    ],
    'physical': [
        'MEDIO FÍSICO__Altitud capital \n(m)',
        'MEDIO FÍSICO__Densidad (hab/km2)',
        'MEDIO FÍSICO__Superficie (km2)',
    ],
    'environment': [
        'MEDIOAMBIENTE__CAPITAL NATURAL__FORESTAL__Superficie forestal \n(% s/ total)',
        'MEDIOAMBIENTE__CAPITAL NATURAL__ESPACIOS PROTEGIDOS__Superficie protegida \n(% s/ total)',
    ],
}

all_candidates = [v for block in CANDIDATE_VARS.values() for v in block]
available_cols = df_master.columns.tolist()
missing_candidates = [v for v in all_candidates if v not in available_cols]

print(f'Total candidate variables: {len(all_candidates)}')
print(f'  - Demographic: {len(CANDIDATE_VARS["demographic"])}')
print(f'  - Economic:    {len(CANDIDATE_VARS["economic"])}')
print(f'  - Services:    {len(CANDIDATE_VARS["services"])}')
print(f'  - Housing:     {len(CANDIDATE_VARS["housing"])}')
print(f'  - Physical:    {len(CANDIDATE_VARS["physical"])}')
print(f'  - Environment: {len(CANDIDATE_VARS["environment"])}')

if missing_candidates:
    print(f'\n⚠ Warning: {len(missing_candidates)} variables not found in df_master:')
    for v in missing_candidates:
        print(f'  - {v}')
else:
    print('\n✓ All candidate variables found in df_master')


In [ ]:
# ── 3.1b Column name inspection helper ───────────────────────────────────────
# Run this cell to explore the exact column names for each thematic block.
# Use it to adjust the CANDIDATE_VARS dictionary above if names don't match.

for block_keyword in ['ECONOMIA', 'SERVICIOS', 'VIVIENDA', 'MEDIO', 'MEDIOAMBIENTE']:
    matching = [c for c in df_master.columns if block_keyword in c]
    print(f'\n=== {block_keyword} ({len(matching)} cols) ===')
    for c in matching:
        print(f'  {c}')

In [ ]:
# ── 3.2 Missing values audit ──────────────────────────────────────────────────
# Compute % missing for each candidate variable in Rural-Remote and Rural-Accessible.
# Variables with >40% missing in either typology are excluded.
#
# NOTE: The 6 binary SI/NO service variables were already imputed (NULL → 0)
# in section 2.7. They will show 0% missing here, which is correct.
# The aridez index is pre-excluded via EXPLICIT_EXCLUDE and not in CANDIDATE_VARS.

found_candidates = [v for v in all_candidates if v in df_master.columns]

# Subset by typology via DuckDB
df_rr = con.execute(f"SELECT * FROM master WHERE tipo_goerlich = '{RURAL_REMOTE}'").df()
df_ra = con.execute(f"SELECT * FROM master WHERE tipo_goerlich = '{RURAL_ACCESSIBLE}'").df()

missing_rr = df_rr[found_candidates].isna().mean() * 100
missing_ra = df_ra[found_candidates].isna().mean() * 100

df_missing = pd.DataFrame({
    'variable': found_candidates,
    'missing_pct_remote':     missing_rr.values,
    'missing_pct_accessible': missing_ra.values,
    'max_missing':            np.maximum(missing_rr.values, missing_ra.values)
})

MISSING_THRESHOLD = 40.0
df_missing['exclude_missing'] = df_missing['max_missing'] > MISSING_THRESHOLD

print(f'Missing audit complete. Threshold: >{MISSING_THRESHOLD}%')
print(f'Excluded (too many missings): {df_missing["exclude_missing"].sum()}')
print(f'Retained: {(~df_missing["exclude_missing"]).sum()}')
print()
excl = df_missing[df_missing['exclude_missing']]
if len(excl) == 0:
    print('No variables excluded — all candidates retained.')
else:
    print('Variables excluded due to missing data:')
    for _, row in excl.iterrows():
        print(f'  {row["variable"].split("__")[-1]}')
        print(f'    Remote: {row["missing_pct_remote"]:.1f}%  |  Accessible: {row["missing_pct_accessible"]:.1f}%')


In [ ]:
# ── 3.3 Spearman correlation with response variables ──────────────────────────
# Computed separately for three strata:
#   rural_remote     : Rural-Remote only
#   rural_accessible : Rural-Accessible only
#   rural_combined   : Both rural typologies
#
# Response variables (binary):
#   reverter_binary  : 1 = Reverses in B,  0 = Structural depopulation
#   dynamiser_binary : 1 = Grows in both,  0 = Structural depopulation
#
# Results inform variable prioritisation in p5b — NOT used to filter here.

retained_vars = df_missing[~df_missing['exclude_missing']]['variable'].tolist()

TYPOLOGY_STRATA = {
    'rural_remote':     [RURAL_REMOTE],
    'rural_accessible': [RURAL_ACCESSIBLE],
    'rural_combined':   [RURAL_REMOTE, RURAL_ACCESSIBLE],
}

corr_all = []

for stratum_name, typology_list in TYPOLOGY_STRATA.items():
    df_stratum = df_master[df_master['tipo_goerlich'].isin(typology_list)].copy()

    df_rev = df_stratum[
        df_stratum['behavioural_group'].isin(['Reverses in B', 'Structural depopulation'])
    ].copy()
    df_rev['reverter_binary'] = (df_rev['behavioural_group'] == 'Reverses in B').astype(int)

    df_dyn = df_stratum[
        df_stratum['behavioural_group'].isin(['Grows in both', 'Structural depopulation'])
    ].copy()
    df_dyn['dynamiser_binary'] = (df_dyn['behavioural_group'] == 'Grows in both').astype(int)

    for var in retained_vars:
        block = next((k for k, v in CANDIDATE_VARS.items() if var in v), 'unknown')

        valid_rev = df_rev[[var, 'reverter_binary']].dropna()
        rho_rev, p_rev = spearmanr(valid_rev[var], valid_rev['reverter_binary']) \
            if len(valid_rev) > 10 else (np.nan, np.nan)

        valid_dyn = df_dyn[[var, 'dynamiser_binary']].dropna()
        rho_dyn, p_dyn = spearmanr(valid_dyn[var], valid_dyn['dynamiser_binary']) \
            if len(valid_dyn) > 10 else (np.nan, np.nan)

        corr_all.append({
            'stratum':        stratum_name,
            'variable':       var,
            'block':          block,
            'rho_reverter':   round(rho_rev, 3) if not np.isnan(rho_rev) else np.nan,
            'p_reverter':     round(p_rev, 4)   if not np.isnan(p_rev)   else np.nan,
            'rho_dynamiser':  round(rho_dyn, 3) if not np.isnan(rho_dyn) else np.nan,
            'p_dynamiser':    round(p_dyn, 4)   if not np.isnan(p_dyn)   else np.nan,
        })

df_corr_all = pd.DataFrame(corr_all)

# Display top 15 by |rho_reverter| per stratum
for stratum in ['rural_remote', 'rural_accessible', 'rural_combined']:
    df_s = df_corr_all[df_corr_all['stratum'] == stratum].sort_values(
        'rho_reverter', key=abs, ascending=False)
    print(f'\nTop 15 |ρ reverter| — {stratum}:')
    print(df_s[['variable', 'block', 'rho_reverter', 'p_reverter',
                 'rho_dynamiser', 'p_dynamiser']].head(15).to_string(index=False))

In [ ]:
# ── 3.4 Collinearity screening (Spearman correlation matrix) ─────────────────
# Build correlation matrix for all candidate variables to detect redundant pairs.
# Threshold: |ρ| > 0.70 → exclude one variable from the pair
# Priority rules:
#   1. Direct measures > composite indices (e.g., Pct_65_plus > Indice_envejecimiento)
#   2. Interpretability > computational convenience

from scipy.stats import spearmanr

# ✅ Nombres correctos con tildes y espacios (consistente con df_master)
df_corr = df_master[
    df_master['tipo_goerlich'].isin([RURAL_REMOTE, RURAL_ACCESSIBLE])
].copy()

all_candidate_cols = [v for block in CANDIDATE_VARS.values() for v in block]
available_candidates = [c for c in all_candidate_cols if c in df_corr.columns]
missing_candidates   = [c for c in all_candidate_cols if c not in df_corr.columns]

if missing_candidates:
    print(f'⚠ {len(missing_candidates)} candidate variables not in df_master:')
    for v in missing_candidates:
        print(f'  - {v}')

print(f'Computing correlation matrix for {len(available_candidates)} variables...')

import numpy as np
n = len(available_candidates)
corr_matrix = np.zeros((n, n))

for i, var1 in enumerate(available_candidates):
    for j, var2 in enumerate(available_candidates):
        if i < j:
            valid = df_corr[[var1, var2]].dropna()
            if len(valid) >= 30:
                rho, _ = spearmanr(valid[var1], valid[var2])
                corr_matrix[i, j] = rho
                corr_matrix[j, i] = rho
        elif i == j:
            corr_matrix[i, j] = 1.0

COLLINEARITY_THRESHOLD = 0.70

high_corr_pairs = []
for i in range(n):
    for j in range(i+1, n):
        if abs(corr_matrix[i, j]) >= COLLINEARITY_THRESHOLD:
            high_corr_pairs.append({
                'var1': available_candidates[i],
                'var2': available_candidates[j],
                'rho':  corr_matrix[i, j]
            })

print(f'\nHigh-correlation pairs (|ρ| ≥ {COLLINEARITY_THRESHOLD}): {len(high_corr_pairs)}')

# ── Automatic exclusion rules for demographic variables ───────────────────────
EXCLUSION_RULES = [
    ('Índice envejecimiento', 'Pct_65_plus',   'Pct_65_plus',
     'Direct measure preferred over composite index'),
    ('Tasa dependencia',      'Pct_0_14',       'Pct_0_14',
     'Direct age group preferred over dependency ratio'),
    ('Tasa dependencia',      'Pct_65_plus',    'Pct_65_plus',
     'Direct age group preferred over dependency ratio'),
    ('Edad media',            'Pct_65_plus',    'Pct_65_plus',
     'Direct age distribution preferred over mean age'),
]

variables_to_exclude = set()
exclusion_log = []

for pair in high_corr_pairs:
    var1, var2, rho = pair['var1'], pair['var2'], pair['rho']
    rule_matched = False

    for pattern1, pattern2, keep_pattern, reason in EXCLUSION_RULES:
        match_fwd = pattern1 in var1 and pattern2 in var2
        match_rev = pattern1 in var2 and pattern2 in var1
        if match_fwd or match_rev:
            if match_fwd:
                keep_var    = var2 if keep_pattern == pattern2 else var1
                exclude_var = var1 if keep_var == var2 else var2
            else:
                keep_var    = var1 if keep_pattern == pattern2 else var2
                exclude_var = var2 if keep_var == var1 else var1

            variables_to_exclude.add(exclude_var)
            exclusion_log.append({'excluded': exclude_var, 'kept': keep_var,
                                   'rho': rho, 'reason': reason})
            print(f'\n✗ EXCLUDE: {exclude_var[:70]}')
            print(f'  ✓ KEEP:    {keep_var[:70]}')
            print(f'  ρ = {rho:+.3f}  |  {reason}')
            rule_matched = True
            break

    if not rule_matched and len(high_corr_pairs) > 0:
        print(f'\n⚠ MANUAL REVIEW: {var1[:50]} vs {var2[:50]}  ρ={rho:+.3f}')

print('\n' + '='*80)
print(f'SUMMARY: {len(variables_to_exclude)} variables excluded due to collinearity')
print('='*80)
for exc in exclusion_log:
    print(f'  ✗ {exc["excluded"][:65]} (ρ={exc["rho"]:+.3f})')

COLLINEAR_DROP = list(variables_to_exclude)


In [ ]:
# ── 3.4b Natural capital: Protected vs. Forested surface correlation ──────────
# Cristina/Elena requested inclusion of % Protected Surface alongside % Forested.
# Decision rule: if ρ(Protected, Forested) < 0.7 → include both
#                if ρ ≥ 0.7 → replace Forested with Protected (higher priority)

import scipy.stats as stats

var_forested = 'MEDIOAMBIENTE__CAPITAL NATURAL__FORESTAL__Superficie forestal \n(% s/ total)'
var_protected = 'MEDIOAMBIENTE__CAPITAL NATURAL__ESPACIOS PROTEGIDOS__Superficie protegida \n(% s/ total)'

# Filter to municipalities with valid data for both variables
df_env = df_master[[var_forested, var_protected]].dropna()

# Compute Spearman correlation
rho_env, p_env = stats.spearmanr(df_env[var_forested], df_env[var_protected])

print('═══════════════════════════════════════════════════════════════════════════')
print('Natural Capital Variables: Protected vs. Forested Surface')
print('═══════════════════════════════════════════════════════════════════════════')
print(f'Spearman ρ: {rho_env:.3f}')
print(f'p-value:    {p_env:.3e}')
print(f'N (valid):  {len(df_env):,} municipalities')
print()

# Decision
COLLINEARITY_THRESHOLD = 0.70

if abs(rho_env) < COLLINEARITY_THRESHOLD:
    print(f'✓ Decision: INCLUDE BOTH variables (|ρ| = {abs(rho_env):.3f} < {COLLINEARITY_THRESHOLD})')
    print('  → Both % Forested and % Protected will enter the analysis')
    env_vars_final = [var_forested, var_protected]
else:
    print(f'⚠ Decision: REPLACE Forested with Protected (|ρ| = {abs(rho_env):.3f} ≥ {COLLINEARITY_THRESHOLD})')
    print('  → Only % Protected will enter the analysis (higher priority per Elena)')
    env_vars_final = [var_protected]
    
    # Update CANDIDATE_VARS['environment'] in place
    CANDIDATE_VARS['environment'] = env_vars_final

print()
print(f'Final environment block: {len(env_vars_final)} variable(s)')
for v in env_vars_final:
    print(f'  - {v}')
print('═══════════════════════════════════════════════════════════════════════════')


In [ ]:
# ── 3.5 Build final selected variable list and export selection log ───────────
# Apply all exclusion decisions:
#   1. Missing values (from section 3.2)
#   2. Collinearity (from section 3.4)
#   3. Protected vs Forested decision (from section 3.4b)

# Start with all candidates
final_candidates = [v for block in CANDIDATE_VARS.values() for v in block]

# Remove variables excluded due to collinearity
if 'COLLINEAR_EXCLUSIONS' in globals() and COLLINEAR_EXCLUSIONS:
    final_candidates = [v for v in final_candidates if v not in COLLINEAR_EXCLUSIONS]
    print(f'Removed {len(COLLINEAR_EXCLUSIONS)} variables due to collinearity')

# Check which variables are actually available in df_master
available_final = [v for v in final_candidates if v in df_master.columns]
missing_final   = [v for v in final_candidates if v not in df_master.columns]

if missing_final:
    print(f'\n⚠ Warning: {len(missing_final)} selected variables not found in df_master:')
    for v in missing_final[:5]:
        print(f'  - {v}')

# ✅ Definir ANTES del export
FINAL_SELECTED_VARS = available_final

print(f'\n' + '='*100)
print('FINAL VARIABLE SELECTION')
print('='*100)
print(f'Total candidate variables:    {len([v for block in CANDIDATE_VARS.values() for v in block])}')
print(f'Excluded (collinearity):      {len(COLLINEAR_EXCLUSIONS) if "COLLINEAR_EXCLUSIONS" in globals() else 0}')
print(f'Available in df_master:       {len(available_final)}')
print(f'Missing from df_master:       {len(missing_final)}')
print('='*100)

# Organize by block
print('\nFinal variables by block:')
for block_name, block_vars in CANDIDATE_VARS.items():
    block_final    = [v for v in block_vars if v in available_final]
    block_excluded = [v for v in block_vars if v in COLLINEAR_EXCLUSIONS] \
                     if 'COLLINEAR_EXCLUSIONS' in globals() else []

    print(f'\n{block_name.upper()}:')
    print(f'  Available: {len(block_final)} variables')
    if block_excluded:
        print(f'  Excluded:  {len(block_excluded)} variables (collinearity)')
    for v in block_final:
        print(f'    ✓ {v[:70]}')
    for v in block_excluded:
        print(f'    ✗ {v[:70]} [EXCLUDED]')

# ── Export selection log for downstream notebooks (p5a_lisa, p5b) ─────────────
all_candidates_full = [v for block in CANDIDATE_VARS.values() for v in block]

df_selection_log = pd.DataFrame({
    'variable': all_candidates_full,
    'block': [next((k for k, v in CANDIDATE_VARS.items() if var in v), 'unknown')
              for var in all_candidates_full],
    'selected': [v in FINAL_SELECTED_VARS for v in all_candidates_full],
    'exclusion_reason': [
        'collinearity'        if ('COLLINEAR_EXCLUSIONS' in globals() and v in COLLINEAR_EXCLUSIONS)
        else 'missing_from_master' if v not in df_master.columns
        else ''
        for v in all_candidates_full
    ]
})

out_selvar = DEMO_DERIV / 'p5a_selected_variables.csv'
df_selection_log.to_csv(out_selvar, sep=';', encoding='utf-8-sig', index=False)

print(f'\nSelection log exported: {out_selvar.name}')
print(f'  Total variables: {len(df_selection_log)}')
print(f'  Selected:        {df_selection_log["selected"].sum()}')
print(f'  Excluded:        {(~df_selection_log["selected"]).sum()}')

print(f'\n✓✓✓ FINAL SELECTED VARIABLES: {len(FINAL_SELECTED_VARS)} ✓✓✓')
print('='*100)

---
## 4 · Descriptive statistics — medians by group and typology

In [ ]:
# ── 4.1 Compute medians per group × typology via DuckDB ───────────────────────
# For each selected variable, compute median by behavioural_group × tipo_goerlich.
# DuckDB handles NULLs in MEDIAN automatically (ignores them, consistent with pandas).
#
# Note: 6 rural municipalities with behavioural_group = None are excluded here.
# These have NaN in var_acum_pct_A or var_acum_pct_B due to missing population
# data at period boundaries — documented in p3 methodology. The master CSV
# (p5a_master_dataset.csv) retains all 8,132 municipalities unmodified.

df_master_analysis = df_master[df_master['behavioural_group'].notna()].copy()
n_excluded = len(df_master) - len(df_master_analysis)
print(f'Municipalities excluded (no behavioural group): {n_excluded}')
print(f'Analysis dataset: {len(df_master_analysis):,} municipalities')

con.register('master', df_master_analysis)

# Build dynamic SQL for medians of all selected variables
# ═══════════════════════════════════════════════════════════════════════════════
# UPDATED: Use FINAL_SELECTED_VARS instead of SELECTED_VARS
# ═══════════════════════════════════════════════════════════════════════════════
if 'FINAL_SELECTED_VARS' not in globals():
    print('⚠ Warning: FINAL_SELECTED_VARS not defined, using all candidates')
    vars_to_use = [v for block in CANDIDATE_VARS.values() for v in block 
                   if v in df_master_analysis.columns]
else:
    vars_to_use = FINAL_SELECTED_VARS

print(f'Computing medians for {len(vars_to_use)} variables')

median_cols_sql = ',\n    '.join(
    [f'MEDIAN("{v}") AS "{v}__median"' for v in vars_to_use]
)

query_medians = f"""
SELECT
    tipo_goerlich,
    behavioural_group,
    COUNT(*) AS n_municipalities,
    {median_cols_sql}
FROM master
WHERE tipo_goerlich IN ('{RURAL_REMOTE}', '{RURAL_ACCESSIBLE}')
GROUP BY tipo_goerlich, behavioural_group
ORDER BY tipo_goerlich, behavioural_group
"""

df_medians = con.execute(query_medians).df()

# Save descriptive stats
out_desc = DEMO_DERIV / 'p5a_descriptive_stats.csv'
df_medians.to_csv(out_desc, sep=';', encoding='utf-8-sig', index=False)
print(f'\nDescriptive stats computed and saved: {out_desc.name}')
print(df_medians[['tipo_goerlich', 'behavioural_group', 'n_municipalities']].to_string(index=False))

---
## 5 · Kruskal-Wallis tests + Dunn post-hoc

**Kruskal-Wallis** is a non-parametric test for differences between k ≥ 2 independent
groups. It tests the null hypothesis that all groups share the same distribution.
It does not assume normality — appropriate for skewed socioeconomic variables in
small rural municipalities.

**Dunn's post-hoc test** with Bonferroni correction identifies *which* pairs of groups
differ significantly when Kruskal-Wallis is significant. Bonferroni is conservative
but preferred for publication (minimises false positives).

Both tests are run separately for Rural-Remote and Rural-Accessible.

In [ ]:
# ── 5.1 Kruskal-Wallis H test — per variable × typology ──────────────────────
# Run KW for each selected variable separately for Rural-Remote and Rural-Accessible.
# Produces df_kw with columns: typology, variable, block, kw_H, kw_p, significant
# This feeds directly into celda 5.2 (Dunn post-hoc).

from scipy.stats import kruskal as kruskal_test

df_master_analysis = df_master[df_master['behavioural_group'].notna()].copy()
print(f'Analysis dataset: {len(df_master_analysis):,} municipalities '
      f'({len(df_master) - len(df_master_analysis)} excluded — no behavioural group)')

kw_rows = []

for typology in RURAL_TYPES:
    df_typ = df_master_analysis[df_master_analysis['tipo_goerlich'] == typology]
    print(f'\n{typology}: {len(df_typ):,} municipalities')

    for var in FINAL_SELECTED_VARS:
        if var not in df_typ.columns:
            continue

        block = next((k for k, v in CANDIDATE_VARS.items() if var in v), 'unknown')

        group_data = {
            g: df_typ[df_typ['behavioural_group'] == g][var].dropna().values
            for g in GROUP_ORDER
        }
        # Skip if any group has fewer than 5 observations
        if any(len(v) < 5 for v in group_data.values()):
            continue

        try:
            H, p = kruskal_test(*group_data.values())
            kw_rows.append({
                'typology':  typology,
                'variable':  var,
                'block':     block,
                'kw_H':      round(H, 4),
                'kw_p':      round(p, 6),
                'significant': p < 0.05,
            })
        except Exception as e:
            print(f'  ✗ KW error [{var[:40]}]: {e}')

df_kw = pd.DataFrame(kw_rows).sort_values('kw_p').reset_index(drop=True)

# Save KW results
out_kw = DEMO_DERIV / 'p5a_kruskal_wallis_results.csv'
df_kw.to_csv(out_kw, sep=';', encoding='utf-8-sig', index=False)

print(f'\n{"="*80}')
print('KRUSKAL-WALLIS RESULTS')
print(f'{"="*80}')
print(f'Total tests:             {len(df_kw)} ({len(FINAL_SELECTED_VARS)} vars × 2 typologies)')
print(f'Significant (p < 0.05):  {df_kw["significant"].sum()} ({df_kw["significant"].mean()*100:.1f}%)')
print(f'Saved: {out_kw.name}')
print(f'{"="*80}\n')

print('Top 20 most significant:\n')
for i, row in df_kw.head(20).iterrows():
    sig = '***' if row['kw_p'] < 0.001 else '**' if row['kw_p'] < 0.01 else '*'
    print(f"  {i+1:2d}. [{row['typology'][:14]}] "
          f"{row['variable'][:55]:55s}  H={row['kw_H']:7.2f}  p={row['kw_p']:.3e} {sig}")


In [ ]:
# ── 5.2 Dunn post-hoc test with Bonferroni correction ─────────────────────────
# Run Dunn for each variable × typology where KW was significant.
# Dunn's test: pairwise Mann-Whitney U with Bonferroni correction.
# Implemented manually using scipy (no dependency on scikit-posthocs needed).

def dunn_bonferroni(data_dict):
    """
    Dunn's pairwise post-hoc test with Bonferroni correction.
    data_dict: {group_name: array_of_values}
    Returns: DataFrame with pairwise comparison results.
    """
    group_names = list(data_dict.keys())
    pairs = list(combinations(group_names, 2))
    n_comparisons = len(pairs)  # Bonferroni denominator
    
    results = []
    for g1, g2 in pairs:
        arr1, arr2 = data_dict[g1], data_dict[g2]
        if len(arr1) < 2 or len(arr2) < 2:
            results.append({'group1': g1, 'group2': g2, 'p_raw': np.nan, 'p_bonf': np.nan, 'significant': False})
            continue
        _, p_raw = stats.mannwhitneyu(arr1, arr2, alternative='two-sided')
        p_bonf = min(p_raw * n_comparisons, 1.0)  # Bonferroni correction
        results.append({
            'group1': g1,
            'group2': g2,
            'p_raw': round(p_raw, 6),
            'p_bonf': round(p_bonf, 6),
            'significant': p_bonf < 0.05
        })
    return pd.DataFrame(results)


# Run Dunn for all significant KW results
sig_kw = df_kw[df_kw['significant']]
dunn_results = []

for _, kw_row in sig_kw.iterrows():
    typology = kw_row['typology']
    var = kw_row['variable']
    block = kw_row.get('block', 'unknown')
    
    df_typ = df_master_analysis[df_master_analysis['tipo_goerlich'] == typology]
    
    data_dict = {
        g: df_typ[df_typ['behavioural_group'] == g][var].dropna().values
        for g in GROUP_ORDER
        if g in df_typ['behavioural_group'].unique()
    }
    
    df_dunn = dunn_bonferroni(data_dict)
    df_dunn['typology'] = typology
    df_dunn['variable'] = var
    df_dunn['block'] = block
    dunn_results.append(df_dunn)

df_dunn_all = pd.concat(dunn_results, ignore_index=True) if dunn_results else pd.DataFrame()

# Save Dunn results
out_dunn = DEMO_DERIV / 'p5a_dunn_results.csv'
df_dunn_all.to_csv(out_dunn, sep=';', encoding='utf-8-sig', index=False)

print(f'Dunn post-hoc tests complete.')
print(f'Pairwise comparisons run: {len(df_dunn_all)}')
print(f'Significant pairs (Bonferroni p < 0.05): {df_dunn_all["significant"].sum() if len(df_dunn_all) > 0 else 0}')
print(f'Saved: {out_dunn.name}')

---
## 6 · Figures — boxplots by variable block

In [ ]:
# ── 6.1 Figure helper functions ───────────────────────────────────────────────

# Variables stored as 0-1 proportion in SIDAMUN → rescale to 0-100 for plotting
PROPORTION_VARS = {
    'SERVICIOS__INTERNET__Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)': 100,
}

# Variables with extreme outliers — cap Y axis at given percentile for legibility
YLIM_PERCENTILE = {
    'SERVICIOS__TRANSPORTE__Parque de vehículos x c/ 100 hab.': 99,
}

# Typology colors — consistent with p3 figures
TYPOLOGY_COLORS = {
    'Rural - Remoto':    '#1e8e42',  # dark green
    'Rural - Accesible': '#73c573',  # light green
}

# English labels — keys are normalised (collapsed whitespace, no \n)
VAR_LABELS_EN = {
    'Renta neta media por persona':                                         'Net income per capita (€)',
    'Renta neta media por hogar':                                           'Net household income (€)',
    'Índice de Gini (%)':                                                   'Gini index (%)',
    'Distribución de la renta P80/P20':                                     'Income ratio P80/P20',
    'Tasa de paro':                                                         'Unemployment rate (%)',
    'Afiliados Régimen Especial (R. E.) T. Autónomos (% s/ Total)':        'Self-employed affiliates (% total)',
    'Contratos indefinidos (% s/ Total)':                                   'Permanent contracts (% total)',
    'Total Empresas':                                                       'Total firms',
    'Pensión Contributiva Media':                                           'Mean contributory pension (€)',
    'Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)':        'Broadband coverage ≥100 Mbps (%)',
    'Consultorio de atención primaria (número)':                            'Primary care centres (n)',
    'Oficina de farmacia (número)':                                         'Pharmacies (n)',
    'Nº centros de Educación Primaria':                                     'Primary schools (n)',
    'Tiempo municipio 5.000 hab. o más, más cercano (minutos)':            'Time to nearest 5,000-hab. town (min)',
    'Tiempo municipio 20.000 hab. o más, más cercano (minutos)':           'Time to nearest 20,000-hab. town (min)',
    'Sucursal bancaria (número)':                                           'Bank branches (n)',
    'Parque de vehículos x c/ 100 hab.':                                   'Vehicles per 100 inhabitants',
    'Viviendas no principales (% s/ total)':                               'Non-primary dwellings (% total)',
    'Tamaño medio del hogar':                                              'Mean household size',
    'Hogares unipersonales (% s/ total)':                                  'Single-person households (% total)',
    'Plazas turísticas x c/ 100 hab.':                                     'Tourist beds per 100 inhabitants',
    'Altitud capital (m)':                                                  'Elevation (m)',
    'Densidad (hab/km2)':                                                   'Population density (hab/km²)',
    'Superficie (km2)':                                                     'Municipal area (km²)',
    'Superficie forestal (% s/ total)':                                     'Forest cover (% total area)',
    'Superficie protegida (% s/ total)':                                    'Protected area (% total area)',
    'Mujeres \n(% hab. s/ total)':                                          'Women (% total population)',
    'Pct_0_14':                                                             'Children 0–14 years (% total population)',
    'Pct_15_29':                                                            'Youth 15–29 years (% total population)',
    'Pct_30_64':                                                            'Adults 30–64 years (% total population)',
    'Pct_65_plus':                                                          'Seniors ≥65 years (% total population)',
    'Pct_Mujeres_0_14':                                                    'Women 0–14 years (% total population)',
    'Pct_Hombres_0_14':                                                    'Men 0–14 years (% total population)',
    'Pct_Mujeres_15_29':                                                   'Women 15–29 years (% total population)',
    'Pct_Hombres_15_29':                                                   'Men 15–29 years (% total population)',
    'Pct_Mujeres_30_64':                                                   'Women 30–64 years (% total population)',
    'Pct_Hombres_30_64':                                                   'Men 30–64 years (% total population)',
    'Pct_Mujeres_65_plus':                                                 'Women ≥65 years (% total population)',
    'Pct_Hombres_65_plus':                                                 'Men ≥65 years (% total population)',
    'Edad media población':                                                 'Mean age (years)',
    'Índice envejecimiento\n(%)':                                           'Aging index (%)',
    'Tasa dependencia \n(%)':                                               'Dependency ratio (%)',
    'Población Nacionalidad Extranjera \n(% hab. s/ total)': 'Foreign nationals (% total population)',
}


def normalise_label(raw):
    """Collapse all whitespace variants (\\n, double spaces) into single spaces."""
    return ' '.join(raw.split())


def wrap_group_label(g):
    """Split group label after first word: 'Grows in both' → 'Grows\\nin both'."""
    parts = g.split(' ', 1)
    return '\n'.join(parts)


def get_significance_label(var, typology, g1, g2):
    """Return Dunn Bonferroni significance annotation for a group pair."""
    if len(df_dunn_all) == 0:
        return ''
    row = df_dunn_all[
        (df_dunn_all['variable'] == var) &
        (df_dunn_all['typology'] == typology) &
        (
            ((df_dunn_all['group1'] == g1) & (df_dunn_all['group2'] == g2)) |
            ((df_dunn_all['group1'] == g2) & (df_dunn_all['group2'] == g1))
        )
    ]
    if len(row) == 0 or not row.iloc[0]['significant']:
        return 'ns'
    p = row.iloc[0]['p_bonf']
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    else:
        return '*'


def plot_block_boxplots(block_name, var_list, typology, df_data, save_path,
                        base_fontsize=12):
    """
    Plot a grid of boxplots for a thematic block.
    One subplot per variable, distributions by behavioural group.
    base_fontsize controls all text — set to 11 for Word figures.
    """
    avail_vars = [v for v in var_list if v in df_data.columns and v in FINAL_SELECTED_VARS]
    if not avail_vars:
        print(f'  No variables available for block {block_name} in {typology}')
        return

    n_vars = len(avail_vars)

    # Resolve typology display info here — used throughout the function
    typology_short = 'Remote' if typology == RURAL_REMOTE else 'Accessible'
    typology_color = TYPOLOGY_COLORS.get(typology, '#333333')

    # Layout: 2×2 for exactly 4 variables, otherwise max 3 columns
    if n_vars == 4:
        ncols = 2
    else:
        ncols = min(3, n_vars)
    nrows = int(np.ceil(n_vars / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten() if n_vars > 1 else [axes]

    df_typ = df_data[df_data['tipo_goerlich'] == typology].copy()

    # Rescale proportion variables to percentage
    for var, scale in PROPORTION_VARS.items():
        if var in df_typ.columns:
            df_typ[var] = df_typ[var] * scale

    groups_present = [g for g in GROUP_ORDER if g in df_typ['behavioural_group'].unique()]

    for ax_idx, var in enumerate(avail_vars):
        ax = axes[ax_idx]

        data_per_group = [
            df_typ[df_typ['behavioural_group'] == g][var].dropna().values
            for g in groups_present
        ]

        bp = ax.boxplot(
            data_per_group,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker='o', markerfacecolor='gray', markersize=2, alpha=0.3),
            medianprops=dict(color='black', linewidth=2)
        )

        for patch, group in zip(bp['boxes'], groups_present):
            patch.set_facecolor(GROUP_COLORS[group])
            patch.set_alpha(0.75)

        # Cap Y axis for variables with extreme outliers
        if var in YLIM_PERCENTILE:
            pct = YLIM_PERCENTILE[var]
            all_vals = np.concatenate([d for d in data_per_group if len(d) > 0])
            if len(all_vals) > 0:
                ymax = np.percentile(all_vals, pct)
                ax.set_ylim(bottom=0, top=ymax * 1.05)
        
        # Fixed Y-axis limit for specific variables (removes single outliers)
        if var in FIXED_YLIM:
            ax.set_ylim(bottom=0, top=FIXED_YLIM[var])
        
        # KW significance annotation
        kw_row = df_kw[(df_kw['variable'] == var) & (df_kw['typology'] == typology)]
        kw_label = ''
        if len(kw_row) > 0:
            p = kw_row.iloc[0]['kw_p']
            if not np.isnan(p):
                kw_label = f'  KW p={p:.4f}' + (' *' if p < 0.05 else '')

        # Normalise label and look up English translation
        raw_label = normalise_label(var.split('__')[-1])
        var_label = VAR_LABELS_EN.get(raw_label, raw_label)

        # Subplot title: variable name + KW p-value
        ax.set_title(f'{var_label}{kw_label}', fontsize=base_fontsize, pad=8)
        ax.set_ylabel(var_label, fontsize=base_fontsize - 1)
        ax.set_xticks(range(1, len(groups_present) + 1))
        ax.set_xticklabels(
            [wrap_group_label(g) for g in groups_present],
            fontsize=base_fontsize
        )
        ax.tick_params(axis='y', labelsize=base_fontsize)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines[['top', 'right']].set_visible(False)

    for ax in axes[n_vars:]:
        ax.set_visible(False)

    legend_patches = [
        mpatches.Patch(facecolor=GROUP_COLORS[g], alpha=0.75, label=g)
        for g in groups_present
    ]
    fig.legend(
        handles=legend_patches,
        loc='lower center',
        ncol=len(groups_present),
        fontsize=base_fontsize,
        frameon=False,
        bbox_to_anchor=(0.5, -0.03)
    )

    # Figure suptitle with typology color
    fig.suptitle(
        f'{block_name.capitalize()} variables — Rural-{typology_short}\n'
        f'Behavioural group distributions (Period B: 2018–2025)',
        fontsize=base_fontsize + 3, fontweight='bold', color=typology_color, y=1.01
    )

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {save_path.name}')

In [ ]:
# ── 6.2 Generate boxplot figures per block × typology ─────────────────────────
# One figure per thematic block × typology combination.
# Naming convention: p5a_boxplot_{block}_{typology}.png
# Variables with extreme outliers — cap Y axis at given percentile for legibility
YLIM_PERCENTILE = {
    'SERVICIOS__TRANSPORTE__Parque de vehículos x c/ 100 hab.': 99,
}

# Fixed Y-axis limits — hard cap for single-outlier variables
FIXED_YLIM = {
    'ECONOMIA__PARADOS__POR SEXO__Tasa de paro': 25,
}
print('Generating boxplot figures...')

for typology in RURAL_TYPES:
    typology_tag = 'remote' if typology == RURAL_REMOTE else 'accessible'
    print(f'\n  Typology: {typology}')
    
    for block_name, var_list in CANDIDATE_VARS.items():
        # Filter to selected variables only
        block_selected = [v for v in var_list if v in FINAL_SELECTED_VARS]
        if not block_selected:
            continue
        
        save_path = FIG_DIR / f'p5a_boxplot_{block_name}_{typology_tag}.png'
        plot_block_boxplots(
            block_name=block_name,
            var_list=block_selected,
            typology=typology,
            df_data=df_master,
            save_path=save_path
        )

print('\nAll figures generated.')

In [ ]:
# ── 6.3 Key figures for paper results section ─────────────────────────────────
# One figure per variable — Rural-Remote (left) vs Rural-Accessible (right).
# Side-by-side comparison highlights typology differences for the same variable.
# Only variables with meaningful signal in at least one typology are included.

BASE_FONTSIZE_KEY = 12  # for Word legibility

KEY_VARS_PAPER = [
    # Economic — strongest signals
    ('economic', 'Net household income (€)',
     'ECONOMIA__RENTAS__Renta neta media por hogar'),
    ('economic', 'Gini index (%)',
     'ECONOMIA__RENTAS__DESIGUALDAD__Índice de Gini (%)'),
    ('economic', 'Income ratio P80/P20',
     'ECONOMIA__RENTAS__DESIGUALDAD__Distribución de la renta P80/P20'),
    ('economic', 'Mean contributory pension (€)',
     'ECONOMIA__PENSIONES CONTRIBUTIVAS__Pensión Contributiva Media'),
    ('economic', 'Self-employed affiliates (% total)',
     'ECONOMIA__AFILIADOS__POR SECTOR__Afiliados Régimen Especial (R. E.) T. Autónomos \n(% s/ Total)'),
    # Services — distance gradient clearest signal
    ('services', 'Time to nearest 5,000-hab. town (min)',
     'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 5.000 hab. o más, más cercano (minutos)'),
    ('services', 'Time to nearest 20,000-hab. town (min)',
     'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 20.000 hab. o más, más cercano (minutos)'),
    ('services', 'Broadband coverage ≥100 Mbps (%)',
     'SERVICIOS__INTERNET__Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)'),
    # Housing
    ('housing', 'Mean household size',
     'VIVIENDA__HOGAR__Tamaño medio del hogar'),
    ('housing', 'Non-primary dwellings (% total)',
     'VIVIENDA__TIPOS DE VIVIENDAS (familiares)__Viviendas no principales (% s/ total)'),
    ('housing', 'Single-person households (% total)',
     'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)'),
    # Physical
    ('physical', 'Population density (hab/km²)',
     'MEDIO FÍSICO__Densidad (hab/km2)'),
    # Environment
    ('environment', 'Forest cover (% total area)',
     'MEDIOAMBIENTE__CAPITAL NATURAL__FORESTAL__Superficie forestal \n(% s/ total)'),
    ('environment', 'Protected area (% total area)',
     'MEDIOAMBIENTE__CAPITAL NATURAL__ESPACIOS PROTEGIDOS__Superficie protegida \n(% s/ total)'),
    # Demographic
    ('demographic', 'Women (% total population)',
     'DEMOGRAFIA__POBLACIÓN POR SEXO__Mujeres \n(% hab. s/ total)'),
    ('demographic', 'Youth 15–29 years (% total)',
     'DEMOGRAFIA__Pct_15_29'),
    ('demographic', 'Children 0–14 years (% total)',
     'DEMOGRAFIA__Pct_0_14'),
    ('demographic', 'Adults 30–64 years (% total)',
     'DEMOGRAFIA__Pct_30_64'),
    ('demographic', 'Seniors ≥65 years (% total)',
     'DEMOGRAFIA__Pct_65_plus'),
     ('demographic', 'Mean age (years)',
     'DEMOGRAFIA__EDAD MEDIA__Edad media población'),
     ('demographic', 'Foreign nationals (% total population)',
     'DEMOGRAFIA__NACIONALIDAD__Población Nacionalidad Extranjera \n(% hab. s/ total)'),
]

print('Generating key figures for paper results section...')
print(f'  Variables: {len(KEY_VARS_PAPER)}')
print(f'  Format: side-by-side Rural-Remote vs Rural-Accessible, fontsize={BASE_FONTSIZE_KEY}')
print()

for block, label_en, var in KEY_VARS_PAPER:

    # Skip if variable not in selected vars
    if var not in FINAL_SELECTED_VARS:
        print(f'  SKIP (not in FINAL_SELECTED_VARS): {label_en}')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.subplots_adjust(wspace=0.15)

    for ax, typology in zip(axes, [RURAL_REMOTE, RURAL_ACCESSIBLE]):

        df_typ = df_master_analysis[
            df_master_analysis['tipo_goerlich'] == typology
        ].copy()

        # Rescale proportion variables
        if var in PROPORTION_VARS:
            df_typ[var] = df_typ[var] * PROPORTION_VARS[var]

        groups_present = [
            g for g in GROUP_ORDER
            if g in df_typ['behavioural_group'].unique()
        ]

        data_per_group = [
            df_typ[df_typ['behavioural_group'] == g][var].dropna().values
            for g in groups_present
        ]

        bp = ax.boxplot(
            data_per_group,
            patch_artist=True,
            showfliers=True,
            flierprops=dict(marker='o', markerfacecolor='gray',
                            markersize=2.5, alpha=0.3),
            medianprops=dict(color='black', linewidth=2.5)
        )

        for patch, group in zip(bp['boxes'], groups_present):
            patch.set_facecolor(GROUP_COLORS[group])
            patch.set_alpha(0.75)

        # Cap Y axis for outlier-heavy variables
        if var in YLIM_PERCENTILE:
            pct = YLIM_PERCENTILE[var]
            all_vals = np.concatenate([d for d in data_per_group if len(d) > 0])
            if len(all_vals) > 0:
                ymax = np.percentile(all_vals, pct)
                ax.set_ylim(bottom=0, top=ymax * 1.05)

        # Fixed Y-axis limit
        if var in FIXED_YLIM:
            ax.set_ylim(bottom=0, top=FIXED_YLIM[var])

        # KW p-value
        kw_row = df_kw[
            (df_kw['variable'] == var) &
            (df_kw['typology'] == typology)
        ]
        kw_label = ''
        if len(kw_row) > 0:
            p = kw_row.iloc[0]['kw_p']
            if not np.isnan(p):
                kw_label = f'  KW p={p:.4f}' + (' *' if p < 0.05 else '')

        # Typology display
        typology_short = 'Rural-Remote' if typology == RURAL_REMOTE else 'Rural-Accessible'
        typology_color = TYPOLOGY_COLORS[typology]

        ax.set_title(
            f'{typology_short}{kw_label}',
            fontsize=BASE_FONTSIZE_KEY + 1, fontweight='bold',
            color=typology_color, pad=10
        )
        ax.set_ylabel(label_en, fontsize=BASE_FONTSIZE_KEY)
        ax.set_xticks(range(1, len(groups_present) + 1))
        ax.set_xticklabels(
            [wrap_group_label(g) for g in groups_present],
            fontsize=BASE_FONTSIZE_KEY
        )
        ax.tick_params(axis='y', labelsize=BASE_FONTSIZE_KEY)
        ax.yaxis.set_major_formatter(
            ticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
        )
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines[['top', 'right']].set_visible(False)

    # Shared legend
    legend_patches = [
        mpatches.Patch(facecolor=GROUP_COLORS[g], alpha=0.75, label=g)
        for g in GROUP_ORDER
    ]
    fig.legend(
        handles=legend_patches,
        loc='lower center',
        ncol=4,
        fontsize=BASE_FONTSIZE_KEY,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04)
    )

    # Shared title
    fig.suptitle(
        label_en,
        fontsize=BASE_FONTSIZE_KEY + 3, fontweight='bold', y=1.02
    )

    # Save
    safe_name = (
        label_en.lower()
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('/', '').replace('%', 'pct').replace('€', 'eur')
        .replace('²', '2').replace('≥', 'ge').replace(',', '')
        .replace('–', '_')
    )
    save_path = FIG_DIR / f'Results_p5a_key_{safe_name}.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓  {label_en}')

print(f'\nKey figures saved to: {FIG_DIR}')

In [ ]:
# ── 6.4 Grouped figures for paper results section ────────────────────────────
# One figure per thematic group — each row is one variable, two panels per row
# (Rural-Remote left, Rural-Accessible right).
# Files prefixed '00_' to sort first in the figures folder.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

# ── Load pre-computed stat CSVs ───────────────────────────────────────────────
DUNN_CSV = ROOT / 'data/demography/derived/paper1/p5a_dunn_results.csv'
KW_CSV   = ROOT / 'data/demography/derived/paper1/p5a_kruskal_wallis_results.csv'

df_dunn = pd.read_csv(DUNN_CSV, sep=';')
df_kw   = pd.read_csv(KW_CSV,   sep=';')

def _safe_float(x):
    try:
        return float(str(x).replace(',', '.'))
    except:
        return np.nan

def lookup_kw_p(typology, var):
    r = df_kw[(df_kw['typology'] == typology) & (df_kw['variable'] == var)]
    return _safe_float(r['kw_p'].values[0]) if not r.empty else np.nan

def lookup_dunn_p(typology, var, g1, g2):
    r = df_dunn[
        (df_dunn['typology'] == typology) &
        (df_dunn['variable'] == var) &
        (((df_dunn['group1'] == g1) & (df_dunn['group2'] == g2)) |
         ((df_dunn['group1'] == g2) & (df_dunn['group2'] == g1)))
    ]
    return _safe_float(r['p_bonf'].values[0]) if not r.empty else np.nan

def fmt_p(p):
    if np.isnan(p): return '—'
    if p < 0.001:   return 'p<0.001'
    if p < 0.01:    return f'p={p:.3f}'
    return f'p={p:.3f}'

def sig_symbol(p, med_key, med_ref):
    if np.isnan(p) or p >= 0.05:
        return 'ns'
    n = 3 if p < 0.001 else (2 if p < 0.01 else 1)
    sym = '>' if med_key >= med_ref else '<'
    return sym * n

def add_stat_box(ax, typology, var, base_fs):
    fs      = base_fs - 1.5
    fs_dunn = base_fs - 2.0

    kw_p   = lookup_kw_p(typology, var)
    kw_txt = f'Kruskal-Wallis: {fmt_p(kw_p)}'
    ax.text(
        0.98, 0.97, kw_txt,
        transform=ax.transAxes, ha='right', va='top',
        fontsize=fs, color='#222222',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#f7f7f7',
                  edgecolor='#bbbbbb', alpha=0.95)
    )

    df_t   = df_master_analysis[df_master_analysis['tipo_goerlich'] == typology]
    med_sd = df_t[df_t['behavioural_group'] == 'Structural depopulation'][var].median()

    lines = ['Dunn (Bonferroni)']
    for g_key, sigla in [('Grows in both', 'GiB'), ('Reverses in B', 'RiB')]:
        p_d     = lookup_dunn_p(typology, var, g_key, 'Structural depopulation')
        med_key = df_t[df_t['behavioural_group'] == g_key][var].median()
        sym     = sig_symbol(p_d, med_key, med_sd)
        p_fmt   = fmt_p(p_d)
        lines.append(f'{sigla} {sym} SD ({p_fmt})' if sym != 'ns' else f'{sigla} vs SD · ns')

    ax.text(
        0.98, 0.80, '\n'.join(lines),
        transform=ax.transAxes, ha='right', va='top',
        fontsize=fs_dunn, color='#333333', linespacing=1.6,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#f7f7f7',
                  edgecolor='#bbbbbb', alpha=0.95)
    )


BASE_FONTSIZE_KEY = 11

# ── Fixed Y-axis limits ───────────────────────────────────────────────────────
FIXED_YLIM = {
    'ECONOMIA__PARADOS__POR SEXO__Tasa de paro': 25,
}

# ── Figure group definitions ──────────────────────────────────────────────────
FIGURE_GROUPS = [
    {
        'filename': '00_Fig1_economic_income_inequality',
        'suptitle': 'Economic conditions — Income and inequality',
        'vars': [
            ('Net household income (€)',
             'ECONOMIA__RENTAS__Renta neta media por hogar', None),
            ('Gini index (%)',
             'ECONOMIA__RENTAS__DESIGUALDAD__Índice de Gini (%)', None),
            ('Income ratio P80/P20',
             'ECONOMIA__RENTAS__DESIGUALDAD__Distribución de la renta P80/P20', None),
        ]
    },
    {
        'filename': '00_Fig2_economic_labour_enterprise',
        'suptitle': 'Economic conditions — Labour market and enterprise',
        'vars': [
            ('Unemployment rate (%)',
             'ECONOMIA__PARADOS__POR SEXO__Tasa de paro', None),
            ('Permanent contracts (% total)',
             'ECONOMIA__CONTRATOS__POR DURACIÓN__Contratos indefinidos \n(% s/ Total)', None),
            ('Self-employed affiliates (% total)',
             'ECONOMIA__AFILIADOS__POR SECTOR__Afiliados Régimen Especial (R. E.) T. Autónomos \n(% s/ Total)', None),
            ('Mean contributory pension (€)',
             'ECONOMIA__PENSIONES CONTRIBUTIVAS__Pensión Contributiva Media', None),
        ]
    },
    {
        'filename': '00_Fig3_services',
        'suptitle': 'Service provision and accessibility',
        'vars': [
            ('Time to nearest 5,000-hab. town (min)',
             'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 5.000 hab. o más, más cercano (minutos)', None),
            ('Time to nearest 20,000-hab. town (min)',
             'SERVICIOS__DISTANCIAS A LOS SERVICIOS MÁS CERCANOS__Tiempo municipio 20.000 hab. o más, más cercano (minutos)', None),
            ('Broadband coverage ≥100 Mbps (%)',
             'SERVICIOS__INTERNET__Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)', None),
        ]
    },
    {
        'filename': '00_Fig4_housing',
        'suptitle': 'Housing characteristics',
        'vars': [
            ('Mean household size',
             'VIVIENDA__HOGAR__Tamaño medio del hogar', None),
            ('Non-primary dwellings (% total)',
             'VIVIENDA__TIPOS DE VIVIENDAS (familiares)__Viviendas no principales (% s/ total)', None),
            ('Single-person households (% total)',
             'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)', None),
        ]
    },
    {
        'filename': '00_Fig5_physical_environment',
        'suptitle': 'Physical environment and natural capital',
        'vars': [
            ('Population density (hab/km²)',
             'MEDIO FÍSICO__Densidad (hab/km2)', 99),
            ('Forest cover (% total area)',
             'MEDIOAMBIENTE__CAPITAL NATURAL__FORESTAL__Superficie forestal \n(% s/ total)', None),
            ('Protected area (% total area)',
             'MEDIOAMBIENTE__CAPITAL NATURAL__ESPACIOS PROTEGIDOS__Superficie protegida \n(% s/ total)', None),
        ]
    },
   {
        'filename': '00_Fig6a_demographic_age_structure',
        'suptitle': 'Demographic structure — Age groups, ratios and foreign nationals',
        'vars': [
            ('Children 0–14 years (% total population)',
             'DEMOGRAFIA__Pct_0_14', None),
            ('Youth 15–29 years (% total population)',
             'DEMOGRAFIA__Pct_15_29', None),
            ('Adults 30–64 years (% total population)',
             'DEMOGRAFIA__Pct_30_64', None),
            ('Seniors ≥65 years (% total population)',
             'DEMOGRAFIA__Pct_65_plus', None),
            ('Mean age (years)',
             'DEMOGRAFIA__EDAD MEDIA__Edad media población', None),
            ('Aging index (%)',
             'DEMOGRAFIA__RATIOS__Índice envejecimiento\n(%)', None),
            ('Dependency ratio (%)',
             'DEMOGRAFIA__RATIOS__Tasa dependencia \n(%)', None),
            ('Foreign nationals (% total population)',
             'DEMOGRAFIA__NACIONALIDAD__Población Nacionalidad Extranjera \n(% hab. s/ total)', None),
        ]
    },
    {
        'filename': '00_Fig6b_demographic_age_by_sex',
        'suptitle': 'Demographic structure — Age groups by sex',
        'vars': [
            ('Women (% total population)',
             'DEMOGRAFIA__POBLACIÓN POR SEXO__Mujeres \n(% hab. s/ total)', None),
            ('Women 0–14 years (% total population)',
             'DEMOGRAFIA__Pct_Mujeres_0_14', None),
            ('Men 0–14 years (% total population)',
             'DEMOGRAFIA__Pct_Hombres_0_14', None),
            ('Women 15–29 years (% total population)',
             'DEMOGRAFIA__Pct_Mujeres_15_29', None),
            ('Men 15–29 years (% total population)',
             'DEMOGRAFIA__Pct_Hombres_15_29', None),
            ('Women 30–64 years (% total population)',
             'DEMOGRAFIA__Pct_Mujeres_30_64', None),
            ('Men 30–64 years (% total population)',
             'DEMOGRAFIA__Pct_Hombres_30_64', None),
            ('Women ≥65 years (% total population)',
             'DEMOGRAFIA__Pct_Mujeres_65_plus', None),
            ('Men ≥65 years (% total population)',
             'DEMOGRAFIA__Pct_Hombres_65_plus', None),
        ]
    },
]

# ── Render loop ───────────────────────────────────────────────────────────────
print('Generating grouped figures for paper results section...')

for fig_group in FIGURE_GROUPS:

    n_vars = len(fig_group['vars'])
    fig, axes = plt.subplots(
        n_vars, 2,
        figsize=(14, 5.5 * n_vars),
        squeeze=False
    )
    fig.subplots_adjust(hspace=0.35, wspace=0.32)

    if 'labour' in fig_group['filename']:
        fig.text(
            0.5, 0.998,
            'Villarroya (La Rioja, n = 5 inhab., unemployment = 100%) '
            'excluded from unemployment plot; retained in all statistical analyses.',
            ha='center', va='top',
            fontsize=BASE_FONTSIZE_KEY - 2, color='#555555', style='italic'
        )

    for row_idx, var_tuple in enumerate(fig_group['vars']):
        label_en = var_tuple[0]
        var      = var_tuple[1]
        cap_pct  = var_tuple[2] if len(var_tuple) > 2 else None

        if var not in FINAL_SELECTED_VARS:
            print(f'  SKIP (not in FINAL_SELECTED_VARS): {label_en}')
            continue

        for col_idx, typology in enumerate([RURAL_REMOTE, RURAL_ACCESSIBLE]):

            ax = axes[row_idx, col_idx]

            df_typ = df_master_analysis[
                df_master_analysis['tipo_goerlich'] == typology
            ].copy()

            if var in PROPORTION_VARS:
                df_typ[var] = df_typ[var] * PROPORTION_VARS[var]

            groups_present = [
                g for g in GROUP_ORDER
                if g in df_typ['behavioural_group'].unique()
            ]

            data_per_group = [
                df_typ[df_typ['behavioural_group'] == g][var].dropna().values
                for g in groups_present
            ]

            show_fliers = var not in FIXED_YLIM
            bp = ax.boxplot(
                data_per_group,
                patch_artist=True,
                showfliers=show_fliers,
                flierprops=dict(marker='o', markerfacecolor='gray',
                                markersize=2.5, alpha=0.3),
                medianprops=dict(color='black', linewidth=2.5)
            )

            for patch, group in zip(bp['boxes'], groups_present):
                patch.set_facecolor(GROUP_COLORS[group])
                patch.set_alpha(0.75)

            if cap_pct is not None:
                all_vals = np.concatenate([d for d in data_per_group if len(d) > 0])
                ymax     = np.percentile(all_vals, cap_pct)
                ax.set_ylim(bottom=0, top=ymax * 1.05)

            if var in YLIM_PERCENTILE:
                pct      = YLIM_PERCENTILE[var]
                all_vals = np.concatenate([d for d in data_per_group if len(d) > 0])
                if len(all_vals) > 0:
                    ymax = np.percentile(all_vals, pct)
                    ax.set_ylim(bottom=0, top=ymax * 1.05)

            if var in FIXED_YLIM:
                ax.set_ylim(bottom=-1, top=FIXED_YLIM[var])

            typology_short = 'Rural-Remote' if typology == RURAL_REMOTE else 'Rural-Accessible'
            typology_color = TYPOLOGY_COLORS[typology]
            if row_idx == 0:
                ax.set_title(
                    typology_short,
                    fontsize=BASE_FONTSIZE_KEY + 2, fontweight='bold',
                    color=typology_color, pad=10
                )

            ax.set_xticklabels(
                [wrap_group_label(g) for g in groups_present],
                fontsize=BASE_FONTSIZE_KEY - 1
            )
            ax.tick_params(axis='y', labelsize=BASE_FONTSIZE_KEY)

            _all_vals = np.concatenate([d for d in data_per_group if len(d) > 0])
            _yrange   = _all_vals.max() - _all_vals.min() if len(_all_vals) > 0 else 1
            if _yrange < 3:
                ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}'))
                ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5, prune='both'))
            else:
                ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
                ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6, prune='both'))

            ax.grid(axis='y', linestyle='--', alpha=0.4)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_ylabel(label_en, fontsize=BASE_FONTSIZE_KEY)

            add_stat_box(ax, typology, var, base_fs=BASE_FONTSIZE_KEY)

    legend_patches = [
        mpatches.Patch(facecolor=GROUP_COLORS[g], alpha=0.75, label=g)
        for g in GROUP_ORDER
    ]
    fig.legend(
        handles=legend_patches,
        loc='lower center',
        ncol=4,
        fontsize=BASE_FONTSIZE_KEY,
        frameon=False,
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.suptitle(
        fig_group['suptitle'],
        fontsize=BASE_FONTSIZE_KEY + 4, fontweight='bold', y=1.01
    )

    save_path = FIG_DIR / f"{fig_group['filename']}.png"
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print(f'  ✓  {fig_group["filename"]}.png')
    plt.show()

print(f'\nAll grouped figures saved to: {FIG_DIR}')

---
## 7 · Export outputs

In [ ]:
# ── 7.1 Export rural analysis dataset (filtered from master) ────────────────
# Filter master to Rural-Remote + Rural-Accessible via DuckDB SQL.
# This is the clean analysis-ready subset passed to p5b.

identity_cols = [
    'Mun_Code', 'Mun_Name', 'Comarca_Code', 'Comarca_Name',
    'Prov_Code', 'Prov_Name', 'CCAA_Code', 'CCAA_Name',
    'tipo_goerlich', 'size_group', 'Pop_ref',
    'var_acum_pct_A', 'var_acum_pct_B',
    'behavioural_group',
    'lisa_I', 'lisa_p_sim', 'lisa_sig_quad'
]

# Build column list for SQL SELECT
final_cols = [c for c in identity_cols if c in df_master.columns] + [
    v for v in FINAL_SELECTED_VARS if v in df_master.columns
]
select_clause = ', '.join([f'"{c}"' for c in final_cols])

# Filter via DuckDB SQL — Rural-Remote and Rural-Accessible only
query_rural = f"""
    SELECT {select_clause}
    FROM master
    WHERE tipo_goerlich IN ('{RURAL_REMOTE}', '{RURAL_ACCESSIBLE}')
    ORDER BY tipo_goerlich, behavioural_group, Mun_Code
"""
df_rural_export = con.execute(query_rural).df()

out_rural = DEMO_DERIV / 'p5a_rural_analysis_dataset.csv'
df_rural_export.to_csv(out_rural, sep=';', encoding='utf-8-sig', index=False)

print('Output files summary:')
outputs = [
    DEMO_DERIV / 'p5a_master_dataset.csv',
    DEMO_DERIV / 'p5a_rural_analysis_dataset.csv',
    DEMO_DERIV / 'p5a_selected_variables.csv',
    DEMO_DERIV / 'p5a_descriptive_stats.csv',
    DEMO_DERIV / 'p5a_kruskal_wallis_results.csv',
    DEMO_DERIV / 'p5a_dunn_results.csv',
]
for f in outputs:
    if f.exists():
        size_kb = f.stat().st_size / 1024
        print(f'  ✓  {f.name}  ({size_kb:.1f} KB)')
    else:
        print(f'  ✗  {f.name}  NOT FOUND')

figs = list(FIG_DIR.glob('*.png'))
print(f'\n  Figures in {FIG_DIR.name}/: {len(figs)} files')
for f in sorted(figs):
    print(f'  ✓  {f.name}')

print(f'\nMaster dataset (all typologies): {len(df_master):,} municipalities')
print(f'Rural analysis dataset: {len(df_rural_export):,} municipalities (Remote + Accessible)')


## 8 · Interpretation notes

---

### Methodological notes

**DuckDB join strategy**: Master dataset built via SQL join on 5-digit `Mun_Code`
(CMUNXL from SIDAMUN), not 3-digit CMUN. This ensures correct municipality-level
matching across all Spanish provinces.

**Variable selection**: 43 variables entered the final analysis (post-collinearity
screening, ρ threshold = 0.70). No variables were excluded by the automatic
collinearity rules, meaning all demographic indices (Índice envejecimiento, Tasa
dependencia, Edad media) were retained alongside the direct age group measures.
The collinearity screening confirmed ρ < 0.70 for all pairs in this dataset.

**Age group computation**: SIDAMUN pyramid columns express each sex as % of their
own sex total (not % of total population). Aggregated age groups were computed by
multiplying each quinquennial column by the corresponding sex weight (% of total
population) before summing: Pct_X = Σ(Pirámide_Mujeres_X × Pct_Mujeres/100) +
Σ(Pirámide_Hombres_X × Pct_Hombres/100). Mean age group total = 97.1% (min 71.5%
for municipalities with missing quinquennial data in SIDAMUN).

**Statistical approach**:
- Kruskal-Wallis H test (non-parametric) across 4 behavioural groups
- Dunn-Bonferroni post-hoc for pairwise comparisons where KW p < 0.05
- 86 total tests (43 vars × 2 typologies); 85/86 significant (98.8%)

**Service variables**: SI/NO columns recoded to 1/0 where no count equivalent
exists (Biblioteca, CRA). Where both SI/NO and número exist, only número retained.

---

### Key findings and hypothesis testing

#### **H2: Natural capital attracts rural immigration**

**Forest cover** (% total area):
- **Rural-Remote**: KW p=0.001. Dynamisers (GiB) > Structural Decline (p=0.020);
  Reverters vs SD not significant. Effect present but weak.
- **Rural-Accessible**: KW p<0.001. Both GiB >> SD (p=0.002) and RiB >> SD
  (p=0.004). Municipalities in demographic recovery consistently show higher
  forest cover than those in structural decline.

**Protected area** (% total area):
- **Rural-Remote**: KW p=0.360 → **not significant**. No differentiation between
  behavioural groups. In Remote typology, protected surface does not discriminate
  recovering from declining municipalities.
- **Rural-Accessible**: KW p=0.030 → significant overall, but Dunn post-hoc
  GiB vs SD and RiB vs SD both ns. The effect is driven by other pairwise
  comparisons (likely Loses in B), not by the recovery groups of interest.

**Conclusion on H2**: Partial support. Forest cover differentiates recovering
from declining municipalities, particularly in Rural-Accessible. Protected area
shows no consistent pattern across recovery groups. The forest cover signal is
stronger in Accessible than Remote typology — consistent with the hypothesis that
natural capital attracts amenity migrants in more accessible locations, where
lifestyle migration is feasible. In Remote areas, high forest cover is ubiquitous
and therefore non-discriminating.

**Methodological note**: Correlation ≠ causation. High forest cover may reflect
historical low development pressure rather than active attraction. Discuss in
Paper 1 Discussion as competing interpretations.

---

#### **H3: Rural immigration is young and feminized**

**Overall assessment**: 98.8% of KW tests significant. Demographic variables
dominate the top of the KW ranking, indicating that age structure is the primary
differentiator between behavioural groups — stronger than any socioeconomic
variable tested.

**% Women (total)**:
- **Rural-Remote**: KW p<0.001 but Dunn GiB vs SD ns, RiB vs SD ns →
  no significant difference in overall sex ratio between recovery and decline groups
- **Rural-Accessible**: KW p<0.001. GiB >>> SD (p<0.001); RiB vs SD ns →
  Dynamisers have significantly higher % women than Structural Decline, but
  Reverters do not differ from SD in overall sex ratio

**Women 15–29 years**:
- **Both typologies**: KW p<0.001. GiB >>> SD (p<0.001), RiB >>> SD (p<0.001) →
  **confirmed across both typologies**. Recovering municipalities have
  significantly higher proportions of young women than declining ones.

**Men 15–29 years**:
- **Both typologies**: KW p<0.001. GiB >>> SD (p<0.001), RiB >>> SD (p<0.001) →
  Young men also significantly higher in recovering municipalities. The youth
  signal is present for **both sexes**, not exclusively women.

**Children 0–14 years** (total, women, men):
- **Both typologies**: KW p<0.001. GiB >>> SD (p<0.001), RiB >>> SD (p<0.001) →
  Strong signal. Recovering municipalities have significantly more children,
  consistent with young family in-migration.

**Adults 30–64 years** (total, women, men):
- **Both typologies**: KW p<0.001. GiB >>> SD (p<0.001), RiB >>> SD (p<0.001) →
  Also significantly higher in recovering municipalities. Working-age adults
  are the dominant demographic driver.

**Seniors ≥65 years** (total, women, men):
- **Both typologies**: KW p<0.001. GiB <<< SD (p<0.001), RiB <<< SD (p<0.001) →
  Significantly lower in recovering municipalities. Structural decline is
  characterised by population ageing.
  
**Foreign nationals (% total population)**:
- **Both typologies**: KW p<0.001. GiB >>> SD (p<0.001), RiB >>> SD (p<0.001) →
  confirmed across both Rural-Remote and Rural-Accessible. Recovering municipalities
  (both Dynamisers and Reverters) have significantly higher proportions of foreign
  nationals than municipalities in structural decline.
- Median values visually higher in Reverters than Dynamisers, particularly in
  Rural-Remote, suggesting international immigration plays a relevant role in
  demographic reversal specifically.
- High variability (wide IQR, extreme outliers up to 80%) reflects heterogeneous
  drivers: agricultural labour immigration in some municipalities vs. lifestyle
  migration in others.
- **Interpretation**: International immigration is a significant component of rural
  demographic recovery. This does not resolve the counterurbanization question
  (origin within Spain unknown) but confirms that rural recovery is partly driven
  by foreign in-migration, consistent with broader European patterns of rural
  repopulation through international flows (Bayona & Gil, 2013; Collantes et al., 2014).

**Conclusion on H3**: **Strongly supported**, but with important nuance.
Recovery is not specifically feminized — both young women AND young men are
significantly more present in recovering municipalities. The dominant pattern is
one of **youthful age structure** (higher 0–14, 15–29, 30–64; lower 65+) rather
than feminization per se. The feminization signal in overall % women is only
significant for Dynamisers in Rural-Accessible. This suggests that:
1. Rural recovery is driven by young families and working-age adults of both sexes
2. The feminization hypothesis requires qualification: if present, it is
   concentrated in the most dynamic municipalities (Dynamisers) and in more
   accessible rural areas
3. The 15–29 female signal specifically (Women 15–29 >>> SD, p<0.001) is
   consistent with selective female out-migration reversing in recovering areas,
   but cannot be confirmed without origin data (EMCR, Step 5c)

---

### Variables requiring special attention in interpretation

**Unemployment rate** (Tasa de paro):
- Villarroya (La Rioja, n=5, 100% unemployment) retained in statistical tests
  but excluded from boxplots (FIXED_YLIM cap at 25%).

**Age group constraint**: Pct_0_14 + Pct_15_29 + Pct_30_64 + Pct_65_plus ≈ 100%.
Only 3 of 4 groups should enter logistic regression models (p5b) to avoid
perfect multicollinearity. Recommended exclusion: Pct_30_64 (least interpretable
for H3 narrative).

**Collinearity within demographic block**: Despite passing the ρ < 0.70 threshold
pairwise, the demographic variables form a compositional system. In PCA/PERMANOVA
(Step 5c), consider using only the 4 total age groups + % Women as input, dropping
the sex-disaggregated age groups to reduce dimensionality.

---

### Next steps for Paper 1

1. **Logistic regression** (p5b, Material Suplementario):
   - Include Pct_15_29, Pct_Mujeres_15_29, Pct_0_14, Pct_65_plus as predictors
   - Exclude Pct_30_64 (compositional constraint)
   - Expected: Pct_15_29 and Pct_0_14 strongest positive predictors of recovery

2. **Multivariate analysis** (PCA + PERMANOVA, new notebook):
   - Demographic variables will likely dominate PC1 (ageing gradient)
   - PC2 may capture socioeconomic differentiation
   - PERMANOVA expected: high R², all group pairs significant

3. **EMCR analysis** (Step 5c):
   - Test whether the young female signal reflects in-migration or reduced
     out-migration
   - Origin data needed to confirm/reject counterurbanization hypothesis

4. **LISA supplementary** (p5a_lisa_spatial_outliers):
   - Test whether HL municipalities show younger age structure than LL
   - Key for Cristina: does spatial resilience correlate with demographic youth?

---

### Data quality checks

✅ 86/86 KW tests ran without errors  
✅ 85/86 significant (p < 0.05) — demographically rich dataset  
✅ Age group totals: mean 97.1%, range 71.5–100% (missings in small municipalities)  
✅ All 43 selected variables found in df_master  
✅ Protected vs. Forested: ρ = 0.457 → both retained (below 0.70 threshold)  
✅ Mun_Code format: 5-digit, 8,132 unique municipalities  

---

**End of p5a notebook**